# 1.1 Demo - Multi Flow SDP with Liquid Clustering and Data Quality
**Course:** Advanced Techniques with Spark Declarative Pipelines · **Lesson:** Introduction to Multi Flows, Expectations and Liquid Clustering in SDP

In this demo you will:
1. Ingest **two independent sources** into **one** Bronze streaming table with **append flows**
2. Enforce **expectations** (`expect` = warn, `expect_or_drop`, `expect_or_fail`) in Silver and keep rejected rows in a **quarantine** table
3. Store Silver and Gold with **Liquid Clustering** (`cluster_by=[...]`) and change the keys without rewriting data
4. Process new files **incrementally**, add a **third source without a full refresh**, and recover from an `expect_or_fail` failure

| Databricks UI | This notebook |
|---|---|
| Pipeline source files in the editor | `01-advanced-sdp/pipelines/multi_flow/transformations/*.py` (open them in VS Code / JupyterLab) |
| *Dry run* | `pipe.validate()` |
| *Start* / *Full refresh all* | `pipe.run()` / `pipe.run(full_refresh=True)` |
| Pipeline settings → Configuration | `configuration={...}` (read in code with `spark.conf.get`) |
| Event log / data quality tab | `pipe.event_log()`, `pipe.expectation_metrics()` |

> The pipeline source files use the real Databricks API (`from pyspark import pipelines as dp`). You can paste them into a Databricks pipeline unchanged (see `databricks/README.md`).

## A. Classroom setup
Equivalent of `%run ./Includes/Classroom-Setup-1.1`. It creates `spark`, `display`, `dbutils`, `DA`, and the `%sql` magic, and gives this lesson its own schema.

In [ ]:
from adelab.classroom import setup
from adelab import datagen

DA = setup("1.1", reset=True)

In [ ]:
SOURCE_A = f"{DA.paths.landing}/source_flow_a"
SOURCE_B = f"{DA.paths.landing}/source_flow_b"
for _ in range(3):                      # three files per landing zone
    datagen.land_events(SOURCE_A, "a")
    datagen.land_events(SOURCE_B, "b")

display(dbutils.fs.ls(SOURCE_A))

## B. Explore the raw data
Newline-delimited JSON. Notice the dirty values the pipeline must handle: `event_ts` is null, `status` is not in the allowed set, `amount` is negative.

In [ ]:
from pyspark.sql import functions as F

raw = spark.read.json(SOURCE_A).unionByName(spark.read.json(SOURCE_B))
display(raw.limit(10))

In [ ]:
display(raw.select(
    F.count("*").alias("rows"),
    F.sum(F.col("event_ts").isNull().cast("int")).alias("null_event_ts"),
    F.sum((~F.col("status").isin("ACTIVE", "INACTIVE", "PENDING", "ERROR")).cast("int")).alias("bad_status"),
    F.sum((F.col("amount") < 0).cast("int")).alias("negative_amount")))

## C. The pipeline source code
Three files, one per layer. Read them before running anything:
- **`01_bronze.py`**: `dp.create_streaming_table("bronze_events")` plus **two** `@dp.append_flow(target="bronze_events")` functions. Each flow has its **own checkpoint**, so you can add, remove, or backfill a source without touching the others.
- **`02_silver.py`**: `@dp.table(cluster_by=["status", "ingest_date"])` with three kinds of expectations, and a quarantine table built from the **inverse** of the drop rules.
- **`03_gold.py`**: a `@dp.materialized_view`, also clustered.

![Pipeline graph: source_flow_a and source_flow_b feed append flows flow_a and flow_b, each with its own checkpoint, into bronze_events; silver_events applies warn, drop and fail expectations and silver_quarantine keeps the dropped rows; gold_daily_metrics aggregates Silver](../images/1.1-multi-flow-architecture.svg)

**How to read it:**
- Left to right is the dataflow graph the dry run below prints: `flow_a` and `flow_b` both target `bronze_events`, each with its own checkpoint.
- `silver_events` and `silver_quarantine` both read `bronze_events`. The quarantine table uses the inverse of the drop rules, so every rejected row is kept somewhere.
- `gold_daily_metrics` is a materialized view over Silver. The clustering keys shown are the `cluster_by` values in the code.

In [ ]:
DA.show_source("pipelines/multi_flow")

## D. Create the pipeline and run a dry run
The configuration is what you would enter under *Pipeline settings → Advanced → Configuration*. The code reads it with `spark.conf.get("source_flow_a")`.

A dry run resolves the dataflow graph without writing data. Note the execution order: the engine infers it from the tables each function reads.

In [ ]:
pipe = DA.pipeline(
    "multi_flow",
    source="pipelines/multi_flow",
    configuration={"source_flow_a": SOURCE_A, "source_flow_b": SOURCE_B},
)
pipe.validate()

## E. Run the first update
The summary below matches the pipeline graph in the Databricks UI: one row per flow, with output rows and data-quality counts.

In [ ]:
update = pipe.run()

## F. Bronze: two flows, one table

In [ ]:
%%sql
SELECT source_flow, count(*) AS rows, count(DISTINCT source_file) AS files
FROM bronze_events
GROUP BY source_flow
ORDER BY source_flow

In [ ]:
# One checkpoint per flow: this isolation is what makes multi-flow tables safe to evolve.
display(dbutils.fs.ls(f"{DA.paths.working_dir}/pipelines/multi_flow/checkpoints/bronze_events"))

## G. Silver: data quality
### G1. Expectation metrics from the event log
In Databricks you would query the event log with SQL:
```sql
SELECT row_expectations.dataset, row_expectations.name,
       SUM(row_expectations.passed_records) AS passed, SUM(row_expectations.failed_records) AS failed
FROM (
  SELECT explode(from_json(details:flow_progress:data_quality:expectations,
         'array<struct<name:string,dataset:string,passed_records:int,failed_records:int>>')) AS row_expectations
  FROM event_log('<pipeline-id>') WHERE event_type = 'flow_progress')
GROUP BY ALL
```
Locally the event log has the same columns. `details` is a JSON string, so we parse it with `from_json`:

In [ ]:
display(pipe.expectation_metrics())

In [ ]:
events = pipe.event_log()
display(events.filter("event_type = 'flow_progress' AND origin.flow_name = 'silver_events'")
              .select("timestamp", "message", "details"))

### G2. Where did every row go?
![One bronze_events row checked by three rule types: expect (warn) keeps it in silver_events, expect_or_drop removes it and silver_quarantine keeps a copy, expect_or_fail stops the update; all three are recorded in the event log](../images/1.1-expectation-actions.svg)

**How to read it:** each lane is one decorator on `silver_events`. The counts in the next cells check the middle column: negative amounts stay in Silver, dropped rows appear in `silver_quarantine`, and the fail lane is what section K triggers.

- **warn** (`non_negative_amount`): the row is kept and counted. Negative amounts are still in Silver.
- **drop** (`valid_timestamp`, `valid_status`): the row is removed from Silver and counted. The quarantine table keeps it, with the names of the rules it broke.
- **fail** (`valid_event_id`): any violation fails the update (section K).

Note that expectations are declared on **datasets**, never on an `@dp.append_flow`. To check the rows of a multi-flow table, pass `expect_all*={...}` to `dp.create_streaming_table()`; the rules then apply to every flow that writes into it.

In [ ]:
%%sql
SELECT 'bronze' AS layer, count(*) AS rows FROM bronze_events
UNION ALL SELECT 'silver', count(*) FROM silver_events
UNION ALL SELECT 'quarantine', count(*) FROM silver_quarantine

In [ ]:
%%sql
SELECT event_id, status, event_ts, amount, failed_expectations
FROM silver_quarantine
ORDER BY event_id
LIMIT 10

In [ ]:
%%sql
-- warn-only rule: violating rows are kept
SELECT count(*) AS negative_amount_rows_in_silver FROM silver_events WHERE amount < 0

## H. Liquid Clustering
`DESCRIBE DETAIL` shows the clustering columns. In Databricks, pipeline tables are clustered by predictive optimization / automatic maintenance. Locally, `pipe.maintenance()` runs `OPTIMIZE`, which clusters the data incrementally.

![Left: Hive-style partitioning by status puts files in one folder per value, with skewed file sizes and a key fixed at creation. Right: Liquid Clustering keeps similar-sized files with narrow key ranges; after changing the keys, old files stay valid and new data uses the new keys](../images/1.1-liquid-clustering.svg)

**How to read it:** `silver_events` uses the right-hand layout. `DESCRIBE DETAIL` shows the keys (`clusteringColumns`), `pipe.maintenance()` is the OPTIMIZE step, and the `ALTER TABLE ... CLUSTER BY (device_id)` below is the "change the keys" arrow: no file is rewritten.

In [ ]:
%%sql
DESCRIBE DETAIL silver_events

In [ ]:
pipe.maintenance()

In [ ]:
%%sql
DESCRIBE HISTORY silver_events

**Change the clustering keys without rewriting data.** In a pipeline you change `cluster_by` in the code. Outside a pipeline the equivalent is `ALTER TABLE ... CLUSTER BY (...)`. Unlike partitioning, existing files stay valid and only new or optimized data uses the new layout.

The next cell runs that `ALTER TABLE` directly on the pipeline table to show what the pipeline does with such a manual change. On Databricks you can't do this: tables managed by a pipeline can only be changed through the pipeline code.

In [ ]:
%%sql
ALTER TABLE silver_events CLUSTER BY (device_id);

In [ ]:
%%sql
DESCRIBE DETAIL silver_events

The pipeline code still says `cluster_by=["status", "ingest_date"]`, so the next update (section J) puts the declared keys back. Declarative pipelines own their tables' layout.

## I. Gold materialized view

In [ ]:
%%sql
SELECT * FROM gold_daily_metrics ORDER BY ingest_date, status

## J. Incremental processing
Land one new file in **flow B only** and start another update. Bronze reads only the new file (checkpoints). Silver processes only the new Bronze rows. The materialized view is recomputed.

![Incremental updates: flow_a finds no new files, flow_b reads one new file, and the added flow_c reads its new folder with a fresh checkpoint; bronze_events appends only those rows, silver_events processes only new Bronze rows, gold_daily_metrics is recomputed](../images/1.1-incremental-processing.svg)

**How to read it:** pale files are already recorded in a flow's checkpoint and are skipped. The next cell is the `flow_b` row (one new file); section J2 adds the dashed `flow_c` row. In both updates `flow_a` reports 0 rows.

In [ ]:
datagen.land_events(SOURCE_B, "b")
update = pipe.run()          # expected: flow_a 0 rows, flow_b 50 rows

`clusteringColumns` is back to `["status", "ingest_date"]`: the update re-applied the keys declared in the code.

In [ ]:
%%sql
DESCRIBE DETAIL silver_events

### J2. Add a third source **without** a full refresh
Add a new file containing one more `@dp.append_flow` to the pipeline sources (`pipelines/multi_flow_extras/flow_c.py`). Existing flows keep their checkpoints, so nothing is re-read.

In [ ]:
DA.show_source("pipelines/multi_flow_extras/flow_c.py")

In [ ]:
SOURCE_C = f"{DA.paths.landing}/source_flow_c"
datagen.land_events(SOURCE_C, "c")

pipe = DA.pipeline(
    "multi_flow",
    source=["pipelines/multi_flow", "pipelines/multi_flow_extras"],
    configuration={"source_flow_a": SOURCE_A, "source_flow_b": SOURCE_B, "source_flow_c": SOURCE_C},
)
update = pipe.run()          # flow_a / flow_b: 0 new rows; flow_c: 50 rows

In [ ]:
%%sql
SELECT source_flow, count(*) AS rows FROM bronze_events GROUP BY source_flow ORDER BY 1

## K. `expect_or_fail` stops the update
Land a file in flow A that contains a row with `event_id = NULL`. Bronze accepts it, because Bronze has no rules. The Silver flow fails, **nothing from that micro-batch is committed**, and every downstream flow is skipped.

In [ ]:
from adelab.sdp import PipelineUpdateFailed

datagen.land_events(SOURCE_A, "a", null_event_id=True)
try:
    pipe.run()
except PipelineUpdateFailed as e:
    print(e)

In [ ]:
display(pipe.event_log().filter("level = 'ERROR'").select("timestamp", "origin.flow_name", "message", "error"))

**Recovering.** The bad record is already in Bronze. Running again fails again, because the Silver checkpoint has not moved past that batch. Your options:
1. Fix the data upstream and do a **full refresh** of the affected tables.
2. **Change the rule** (for example, from fail to drop) and run again. The failed micro-batch is processed again under the new rule.

This pipeline reads its policy from a setting (`event_id_policy`). Change the setting and start a new update:

In [ ]:
pipe = DA.pipeline(
    "multi_flow",
    source=["pipelines/multi_flow", "pipelines/multi_flow_extras"],
    configuration={"source_flow_a": SOURCE_A, "source_flow_b": SOURCE_B, "source_flow_c": SOURCE_C,
                   "event_id_policy": "drop"},
)
update = pipe.run()

In [ ]:
%%sql
SELECT (SELECT count(*) FROM bronze_events WHERE event_id IS NULL) AS null_ids_in_bronze,
       (SELECT count(*) FROM silver_events WHERE event_id IS NULL) AS null_ids_in_silver

## L. Full refresh
A full refresh truncates every table, deletes all checkpoints, and reprocesses all source files from the beginning. Use it when a streaming table's logic changes in a way that old data must be recomputed. (Exception: **sinks** are not truncated. Their flows restart from the beginning and append everything again; see demo 1.2.)

Expected: Bronze ends up with the same row count as before the refresh, because every landed file is read again.

In [ ]:
update = pipe.run(full_refresh=True)

## M. Clean up
Equivalent of `DA.cleanup()`. Skip this cell if you want to keep exploring the tables.

In [ ]:
DA.cleanup()